# 第四章 RNN / LSTM 旋律生成与五声音阶约束

本 Notebook 对应书中 **RNN 与 LSTM** 与 **约束生成：五声音阶限制采样**。

1. 从本地按艺术家（artist）目录整理、但来源映射未核实的 MIDI 子集中筛选两类单声部旋律近似。
2. 将 MIDI 音符转成直观的 `(pitch, duration)` token。
3. 用 1 层 LSTM 预测下一个 token。
4. 分别生成自由采样与五声音阶约束采样的 MIDI。
5. 用灰度图比较生成结果的音高轮廓、钢琴卷帘和统计指标。


## 0. 环境与配置

本节集中放置所有可调参数。若仅检查 Notebook 能否运行，可先把 `MAX_FILES_PER_STYLE`、`TRAINING_EPOCHS`、`GENERATED_TOKEN_COUNT` 调小。


In [ ]:
import os
from pathlib import Path

# Matplotlib 在某些受限环境中不能写入 ~/.matplotlib。
# 这里先定位项目根目录，再把缓存固定到 CODE/chapter04，避免从不同 cwd 运行时到处生成缓存目录。
current_path = Path.cwd().resolve()
_project_root_for_cache = current_path
while not (_project_root_for_cache / 'CODE' / 'datasets').exists() and _project_root_for_cache.parent != _project_root_for_cache:
    _project_root_for_cache = _project_root_for_cache.parent
_chapter_code_dir_for_cache = _project_root_for_cache / 'CODE' / 'chapter04'
if not _chapter_code_dir_for_cache.exists():
    _chapter_code_dir_for_cache = current_path
os.environ.setdefault('MPLCONFIGDIR', str(_chapter_code_dir_for_cache / '.matplotlib_cache'))

import math
import random
import re
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pretty_midi

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from IPython.display import Audio, Markdown, display

# 中文字体与灰度友好绘图设置
plt.rcParams['font.sans-serif'] = [
    'PingFang SC', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei',
    'Arial Unicode MS', 'Noto Sans CJK SC', 'DejaVu Sans',
]
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['text.color'] = 'black'
plt.rcParams['savefig.facecolor'] = 'white'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def _is_integer_scalar(value):
    return isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_))


def _expect_value_error(function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except ValueError:
        return
    raise AssertionError(f'{function.__name__} 对非法输入未抛出 ValueError')

# 数据与输出路径
current_path = Path.cwd().resolve()
project_root = current_path
while not (project_root / 'CODE' / 'datasets').exists() and project_root.parent != project_root:
    project_root = project_root.parent

DATASET_DIR = project_root / 'CODE' / 'datasets'
LMD_DIR = DATASET_DIR / 'lmd_clean_midi'
CHAPTER_CODE_DIR = project_root / 'CODE' / 'chapter04'
FIGURES_DIR = CHAPTER_CODE_DIR / 'output_figures'
MIDI_OUTPUT_DIR = CHAPTER_CODE_DIR / 'output_midi'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MIDI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 数据筛选参数
MAX_FILES_PER_STYLE = 36          # 每个风格最多解析多少个 MIDI 文件；想更快可设为 12
MIN_NOTES_PER_MELODY = 64         # 太短的轨道不用于训练
MAX_NOTES_PER_MELODY = 512        # 太长的轨道截断，避免少数长曲主导训练
MAX_DURATION_UNITS = 16            # 音符/休止最长保留 4 个四分音符时值；更长值合并到上界 token
MAX_OVERLAP_RATIO = 0.20          # 重叠音比例过高说明不是清晰单声部
MIN_MEDIAN_PITCH = 48             # 中位音高过低时大概率是 bass
QUANTIZATION_STEP_BEATS = 0.25    # 1 个 duration unit = 十六分音符
WINDOW_LENGTH = 32                # 输入最近 32 个 token
WINDOW_STRIDE = 8                 # 训练窗口步长，避免窗口过度重复
VALIDATION_RATIO = 0.15

# 小模型参数：CPU 也能运行
MODEL_KIND = 'lstm'               # 可改为 'rnn' 观察简单 RNN
EMBEDDING_DIM = 64
HIDDEN_SIZE = 128
DROPOUT_RATE = 0.10
BATCH_SIZE = 64
TRAINING_EPOCHS = 6
LEARNING_RATE = 1e-3

# 生成参数
GENERATED_TOKEN_COUNT = 96
DEFAULT_TEMPERATURE = 1.0
TEMPERATURE_VALUES = [0.5, 1.0, 2.0]
PENTATONIC_PITCH_CLASSES = {0, 2, 4, 7, 9}  # C D E G A，C宫五声
DEFAULT_TEMPO = 120
RENDER_AUDIO_PREVIEW = False      # 默认关闭音频合成，需要试听时可改为 True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('项目根目录:', project_root)
print('本地按艺术家整理的 MIDI 子集（与官方 LMD 的派生关系未核实）:', LMD_DIR)
print('图片输出目录:', FIGURES_DIR)
print('MIDI 输出目录:', MIDI_OUTPUT_DIR)
print('PyTorch device:', DEVICE)


## 1. 数据子集：古典/浪漫旋律 vs 流行主旋律

- **古典/浪漫旋律**：优先选择 Mozart、Beethoven、Chopin、Brahms、Debussy 等目录中的旋律性轨道。
- **流行主旋律**：优先选择 The Beatles、ABBA、Madonna、Michael Jackson、Whitney Houston 等目录中名称带有 `melody / lead / vocal / voice / solo` 倾向的轨道；如果 MIDI 轨道没有清晰命名，则用音区、音符数量、重叠比例做近似筛选。

各艺术家目录采用确定性的轮转抽样，避免只取列表前部目录。输出统一写成钢琴 MIDI，便于比较本次运行中的符号旋律与节奏。结果不代表对应时代或流派总体。


In [ ]:
STYLE_ARTIST_FOLDERS = {
    'classical_romantic': [
        'Wolfgang Amadeus Mozart',
        'Ludwig van Beethoven',
        'Chopin Frederic',
        'Johannes Brahms',
        'Claude Debussy',
        'Haydn',
        'George Frideric Handel',
    ],
    'pop_lead': [
        'The Beatles',
        'ABBA',
        'Madonna',
        'Michael Jackson',
        'Britney Spears',
        'Whitney Houston',
        'Celine Dion',
        'Elton John',
        'Billy Joel',
        'Carpenters',
        'Backstreet Boys',
        'Westlife',
        'Tori Amos',
    ],
}

STYLE_DISPLAY_NAMES = {
    'classical_romantic': '古典/浪漫旋律',
    'pop_lead': '流行主旋律',
}

POSITIVE_TRACK_NAME_KEYWORDS = {
    'melody', 'lead', 'vocal', 'voice', 'singer', 'topline', 'solo', 'right', 'rh',
}
NEGATIVE_TRACK_NAME_KEYWORDS = {
    'bass', 'drum', 'perc', 'chord', 'pad', 'accomp', 'rhythm', 'strum', 'left', 'lh',
}

PC_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
REST_PITCH = None

@dataclass
class MelodySequence:
    """一条用于训练的单声部旋律近似。"""
    style_label: str
    source_path: Path
    track_name: str
    instrument_program: int
    tokens: list
    note_count: int
    overlap_ratio: float
    median_pitch: float


def list_midi_files_for_artists(dataset_dir, artist_folders, max_files):
    """对各 artist 目录做确定性的轮转抽样，避免列表前部目录垄断子集。"""
    artist_file_lists = []
    for artist_index, artist_folder in enumerate(artist_folders):
        artist_dir = dataset_dir / artist_folder
        if not artist_dir.exists():
            print(f'  未找到目录，跳过: {artist_folder}')
            continue
        artist_files = sorted(
            list(artist_dir.rglob('*.mid')) + list(artist_dir.rglob('*.midi')),
            key=lambda path: path.as_posix().lower(),
        )
        artist_rng = random.Random(SEED + artist_index)
        artist_rng.shuffle(artist_files)
        if artist_files:
            artist_file_lists.append(artist_files)

    collected_files = []
    round_index = 0
    while len(collected_files) < max_files:
        added_this_round = False
        for artist_files in artist_file_lists:
            if round_index < len(artist_files):
                collected_files.append(artist_files[round_index])
                added_this_round = True
                if len(collected_files) == max_files:
                    break
        if not added_this_round:
            break
        round_index += 1
    return collected_files


def note_overlap_ratio(notes):
    """估计重叠音比例；比例越高，越不像单声部旋律。"""
    if len(notes) < 2:
        return 0.0
    sorted_notes = sorted(notes, key=lambda note: (note.start, -note.pitch))
    overlapping_notes = 0
    current_end = sorted_notes[0].end
    for note in sorted_notes[1:]:
        if note.start < current_end - 1e-4:
            overlapping_notes += 1
        current_end = max(current_end, note.end)
    return overlapping_notes / max(1, len(sorted_notes) - 1)


def choose_highest_note_per_onset(notes, midi_object):
    """把同一量化 onset 上的和声音压成最高音，得到单声部近似。"""
    onset_groups = {}
    for note in notes:
        onset_beats = midi_object.time_to_tick(note.start) / midi_object.resolution
        onset_index = int(round(onset_beats / QUANTIZATION_STEP_BEATS))
        previous_note = onset_groups.get(onset_index)
        if previous_note is None or note.pitch > previous_note.pitch:
            onset_groups[onset_index] = note
    return [onset_groups[index] for index in sorted(onset_groups)]


def time_span_to_units(midi_object, start_seconds, end_seconds):
    """依据 MIDI tempo map 把秒数区间换算为四分音符时值数，再量化为单位。"""
    tick_span = midi_object.time_to_tick(end_seconds) - midi_object.time_to_tick(start_seconds)
    quarter_beats = tick_span / midi_object.resolution
    return quarter_beats / QUANTIZATION_STEP_BEATS


def track_name_has_keyword(track_name, keywords):
    """按单词匹配轨道提示词；rh/lh 等短词只允许完整匹配。"""
    words = re.findall(r'[a-z0-9]+', (track_name or '').lower())
    for keyword in keywords:
        if len(keyword) <= 2 and keyword in words:
            return True
        if len(keyword) > 2 and any(word == keyword or word.startswith(keyword) for word in words):
            return True
    return False


def instrument_name_score(instrument, style_label):
    """根据轨道名给旋律候选轨道打分；命名只是辅助，不作为绝对规则。"""
    track_name = (instrument.name or '').lower()
    score = 0.0
    if track_name_has_keyword(track_name, POSITIVE_TRACK_NAME_KEYWORDS):
        score += 4.0
    if track_name_has_keyword(track_name, NEGATIVE_TRACK_NAME_KEYWORDS):
        score -= 5.0
    if style_label == 'classical_romantic' and instrument.program in range(0, 8):
        score += 1.0  # 钢琴类音色常用于古典/浪漫 MIDI
    return score


def instrument_to_tokens(instrument, midi_object):
    """转为 (pitch, duration_units)；最多 512 音，长于 16 单位的音符/休止合并到上界。"""
    notes = [note for note in instrument.notes if note.end > note.start]
    if len(notes) < MIN_NOTES_PER_MELODY:
        return []

    melody_notes = choose_highest_note_per_onset(notes, midi_object)
    melody_notes = sorted(melody_notes, key=lambda note: note.start)

    tokens = []
    previous_end = None
    for note in melody_notes[:MAX_NOTES_PER_MELODY]:
        if previous_end is not None:
            rest_units_float = time_span_to_units(midi_object, previous_end, note.start)
            if rest_units_float >= 0.75:
                rest_units = max(1, int(round(rest_units_float)))
                tokens.append((REST_PITCH, min(rest_units, MAX_DURATION_UNITS)))
        duration_units = max(1, int(round(time_span_to_units(midi_object, note.start, note.end))))
        tokens.append((int(note.pitch), min(duration_units, MAX_DURATION_UNITS)))
        previous_end = max(previous_end or note.end, note.end)
    return tokens


def score_melody_candidate(instrument, tokens, overlap_ratio, median_pitch, style_label):
    """综合音符数、音区、重叠比例和轨道名，给候选旋律轨道打分。"""
    note_tokens = [token for token in tokens if token[0] is not REST_PITCH]
    note_count = len(note_tokens)
    score = min(note_count, 256) / 64.0
    score += instrument_name_score(instrument, style_label)
    score -= overlap_ratio * 8.0
    if median_pitch < MIN_MEDIAN_PITCH:
        score -= 3.0
    if 55 <= median_pitch <= 84:
        score += 1.0
    return score


def extract_best_melody_sequence(midi_path, style_label):
    """从一个 MIDI 文件中选出最像单声部旋律的轨道。"""
    try:
        midi_object = pretty_midi.PrettyMIDI(str(midi_path))
    except Exception:
        return None

    best_candidate = None
    best_score = -float('inf')

    for instrument in midi_object.instruments:
        if instrument.is_drum:
            continue
        # 轨道名明确指向低音、打击乐或伴奏时，不把它包装成“主旋律”。
        # 这比只降分更严格，可避免在没有理想候选时仍选中 Double Bass / Drums。
        if track_name_has_keyword(instrument.name or '', NEGATIVE_TRACK_NAME_KEYWORDS):
            continue
        raw_notes = [note for note in instrument.notes if note.end > note.start]
        if len(raw_notes) < MIN_NOTES_PER_MELODY:
            continue
        overlap_ratio = note_overlap_ratio(raw_notes)
        if overlap_ratio > MAX_OVERLAP_RATIO:
            continue
        median_pitch = float(np.median([note.pitch for note in raw_notes]))
        if median_pitch < MIN_MEDIAN_PITCH:
            continue

        tokens = instrument_to_tokens(instrument, midi_object)
        note_tokens = [token for token in tokens if token[0] is not REST_PITCH]
        if len(note_tokens) < MIN_NOTES_PER_MELODY:
            continue

        candidate_score = score_melody_candidate(
            instrument=instrument,
            tokens=tokens,
            overlap_ratio=overlap_ratio,
            median_pitch=median_pitch,
            style_label=style_label,
        )
        if candidate_score > best_score:
            best_score = candidate_score
            best_candidate = MelodySequence(
                style_label=style_label,
                source_path=midi_path,
                track_name=instrument.name or '(unnamed track)',
                instrument_program=int(instrument.program),
                # instrument_to_tokens 已按最多 MAX_NOTES_PER_MELODY 个音符截断；
                # 这里保留其间产生的休止 token，避免再按 token 数二次截断。
                tokens=tokens,
                note_count=len(note_tokens),
                overlap_ratio=overlap_ratio,
                median_pitch=median_pitch,
            )
    return best_candidate

print('数据筛选与 token 化函数定义完毕。')


### 1.1 执行数据筛选

这一单元会解析若干本地 MIDI 文件。若运行时间过长，先调小 `MAX_FILES_PER_STYLE`。


In [ ]:
def collect_style_melodies(style_label):
    """收集一个风格子集中的旋律序列。"""
    midi_files = list_midi_files_for_artists(
        dataset_dir=LMD_DIR,
        artist_folders=STYLE_ARTIST_FOLDERS[style_label],
        max_files=MAX_FILES_PER_STYLE,
    )
    print(f'\n{STYLE_DISPLAY_NAMES[style_label]}: 准备解析 {len(midi_files)} 个 MIDI 文件')

    melody_sequences = []
    for file_index, midi_path in enumerate(midi_files, start=1):
        melody_sequence = extract_best_melody_sequence(midi_path, style_label)
        if melody_sequence is not None:
            melody_sequences.append(melody_sequence)
        if file_index % 10 == 0 or file_index == len(midi_files):
            print(f'  已解析 {file_index:>3}/{len(midi_files)}，保留 {len(melody_sequences)} 条旋律')
    return melody_sequences

style_to_melodies = {
    style_label: collect_style_melodies(style_label)
    for style_label in STYLE_ARTIST_FOLDERS
}

data_summary_rows = []
for style_label, melody_sequences in style_to_melodies.items():
    note_counts = [sequence.note_count for sequence in melody_sequences]
    overlap_ratios = [sequence.overlap_ratio for sequence in melody_sequences]
    median_pitches = [sequence.median_pitch for sequence in melody_sequences]
    data_summary_rows.append({
        '风格子集': STYLE_DISPLAY_NAMES[style_label],
        '旋律条数': len(melody_sequences),
        '平均音符数': round(float(np.mean(note_counts)), 1) if note_counts else 0,
        '平均重叠比例': round(float(np.mean(overlap_ratios)), 3) if overlap_ratios else 0,
        '中位音高均值': round(float(np.mean(median_pitches)), 1) if median_pitches else 0,
    })

data_summary = pd.DataFrame(data_summary_rows)
display(data_summary)

sample_rows = []
for style_label, melody_sequences in style_to_melodies.items():
    for sequence in melody_sequences[:5]:
        sample_rows.append({
            '风格子集': STYLE_DISPLAY_NAMES[style_label],
            '文件': sequence.source_path.name,
            '轨道名': sequence.track_name,
            '音符数': sequence.note_count,
            '重叠比例': round(sequence.overlap_ratio, 3),
            '中位音高': round(sequence.median_pitch, 1),
        })

display(pd.DataFrame(sample_rows))


## 2. 从 MIDI 到 `(pitch, duration)` token

每个 token 是一个二元组：

```text
(pitch, duration_units)
```

- `pitch` 是 MIDI pitch，例如 60 表示中央 C；休止用 `REST` 表示。
- `duration_units` 以四分音符时值为参照：`1` 个单位等于四分音符时值的四分之一（即十六分音符时值），`4` 个单位等于一个四分音符时值。换算依据 MIDI tempo map 的 tick 位置，而不是 `estimate_tempo()`；这里不按拍号分母重新定义单位。
- 本 Notebook 使用两类数据的**并集词表**（union vocabulary），这样两类模型可以从同一个起始动机生成，便于对比。为避免验证集中出现无法编码的 token，词表在文件级切分前由全部保留旋律建立；因此验证文件没有参与参数更新，但验证损失不是“连词表也完全未见”的严格外部评估。


In [ ]:
def token_to_text(token):
    """把 token 转成适合展示的短文本。"""
    pitch, duration_units = token
    if pitch is REST_PITCH:
        return f'REST_D{duration_units}'
    pitch_class_name = PC_NAMES[pitch % 12]
    octave = pitch // 12 - 1
    return f'{pitch_class_name}{octave}_D{duration_units}'


def build_token_vocabulary(style_to_melodies):
    """建立 union vocabulary，并按出现频次排序。"""
    token_counter = Counter()
    for melody_sequences in style_to_melodies.values():
        for sequence in melody_sequences:
            token_counter.update(sequence.tokens)

    if not token_counter:
        raise RuntimeError('没有提取到可训练的 token。请检查 LMD_DIR 或调低筛选阈值。')

    sorted_tokens = [token for token, _ in token_counter.most_common()]
    token_to_id = {token: token_id for token_id, token in enumerate(sorted_tokens)}
    id_to_token = {token_id: token for token, token_id in token_to_id.items()}
    return token_to_id, id_to_token, token_counter


token_to_id, id_to_token, token_counter = build_token_vocabulary(style_to_melodies)

vocabulary_summary = pd.DataFrame({
    '项目': ['词表大小', '总 token 数', '休止 token 数'],
    '数值': [
        len(token_to_id),
        sum(token_counter.values()),
        sum(count for token, count in token_counter.items() if token[0] is REST_PITCH),
    ],
})
display(vocabulary_summary)

most_common_token_rows = []
for token, count in token_counter.most_common(12):
    most_common_token_rows.append({
        'token': token_to_text(token),
        '出现次数': count,
        '音级': 'REST' if token[0] is REST_PITCH else PC_NAMES[token[0] % 12],
        'duration_units': token[1],
    })
display(pd.DataFrame(most_common_token_rows))


## 3. logits、softmax、温度与五声音阶掩码的数值关系

训练好的 LSTM 每一步先输出一组 `logits`，它们只是原始分数，不是概率。`softmax` 将分数转成概率；温度参数改变概率分布的尖锐程度；五声音阶掩码再把不合规则的候选 token 概率置零并重新归一化。


In [ ]:
def softmax_numpy(logits, temperature=1.0):
    """NumPy 版本 softmax，用于绘制概率与温度、mask 的关系。"""
    temperature = float(temperature)
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError('temperature 必须是有限正数。')
    scaled_logits = np.asarray(logits, dtype=float)
    if scaled_logits.ndim != 1 or scaled_logits.size == 0:
        raise ValueError('logits 必须是一维非空向量。')
    if not np.all(np.isfinite(scaled_logits)):
        raise ValueError('logits 必须只包含有限数。')
    scaled_logits = scaled_logits / temperature
    if not np.all(np.isfinite(scaled_logits)):
        raise ValueError('logits / temperature 必须保持有限。')
    scaled_logits = scaled_logits - np.max(scaled_logits)
    probabilities = np.exp(scaled_logits)
    return probabilities / probabilities.sum()


def token_allowed_by_pentatonic(token, allowed_pitch_classes=PENTATONIC_PITCH_CLASSES):
    """判断 token 是否符合五声音阶约束；休止始终允许。"""
    pitch, _ = token
    if pitch is REST_PITCH:
        return True
    return (pitch % 12) in allowed_pitch_classes


def apply_pentatonic_mask(probabilities, candidate_tokens, allowed_pitch_classes=PENTATONIC_PITCH_CLASSES):
    """把非五声音级 token 概率置零，再重新归一化。"""
    probabilities = np.asarray(probabilities, dtype=float)
    candidate_tokens = list(candidate_tokens)
    if (probabilities.ndim != 1 or len(probabilities) != len(candidate_tokens) or
            len(candidate_tokens) == 0):
        raise ValueError('probabilities 与 candidate_tokens 必须是一一对应的非空一维序列。')
    if not np.all(np.isfinite(probabilities)) or np.any(probabilities < 0):
        raise ValueError('probabilities 必须只包含有限非负数。')
    mask = np.array([
        token_allowed_by_pentatonic(token, allowed_pitch_classes)
        for token in candidate_tokens
    ], dtype=float)
    masked_probabilities = probabilities * mask
    probability_sum = masked_probabilities.sum()
    if not np.isfinite(probability_sum) or probability_sum <= 0:
        raise ValueError('候选集中没有任何满足五声音级约束且概率非零的 token。')
    return masked_probabilities / probability_sum

# 用高频 token 做一个小示例，避免图太拥挤。
demo_tokens = [token for token, _ in token_counter.most_common(8)]
demo_logits = np.array([2.4, 1.8, 1.2, 0.8, 0.4, 0.1, -0.2, -0.6])[:len(demo_tokens)]
demo_probabilities = softmax_numpy(demo_logits, temperature=1.0)
demo_masked_probabilities = apply_pentatonic_mask(demo_probabilities, demo_tokens)
assert np.isclose(demo_probabilities.sum(), 1.0)
assert np.isclose(demo_masked_probabilities.sum(), 1.0)
_expect_value_error(softmax_numpy, [0.0, 1.0], temperature=0)
_expect_value_error(softmax_numpy, [0.0, 1.0], temperature=float('nan'))
_expect_value_error(softmax_numpy, [0.0, float('inf')], temperature=1.0)
_expect_value_error(apply_pentatonic_mask, [-0.1, 1.1], demo_tokens[:2])

def plot_softmax_mask_demo(candidate_tokens, logits, probabilities, masked_probabilities):
    labels = [token_to_text(token) for token in candidate_tokens]
    x_positions = np.arange(len(labels))
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), constrained_layout=True)

    axes[0].bar(x_positions, logits, color='0.35', edgecolor='black')
    axes[0].set_title('logits：原始分数')
    axes[0].set_ylabel('分数')

    axes[1].bar(x_positions, probabilities, color='0.55', edgecolor='black', hatch='//')
    axes[1].set_title('softmax 后：概率')
    axes[1].set_ylabel('概率')

    axes[2].bar(x_positions, masked_probabilities, color='0.75', edgecolor='black', hatch='xx')
    axes[2].set_title('五声音阶掩码后：重新归一化')
    axes[2].set_ylabel('概率')

    for axis in axes:
        axis.set_xticks(x_positions)
        axis.set_xticklabels(labels, rotation=45, ha='right')
        axis.grid(axis='y', color='0.85', linewidth=0.8)
        axis.set_axisbelow(True)

    output_path = FIGURES_DIR / 'fig_lstm_softmax_mask_demo.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('图片已保存:', output_path)

plot_softmax_mask_demo(demo_tokens, demo_logits, demo_probabilities, demo_masked_probabilities)


## 4. 训练样本：前 32 个 token 预测第 33 个 token

RNN/LSTM 的训练任务可以理解为：给模型一段前文，让它预测下一个音乐事件。

```text
输入：token[t], token[t+1], ..., token[t+31]
目标：token[t+32]
```

这里按乐曲划分训练集与验证集，再从每首乐曲内部切窗口，避免同一首乐曲的相邻窗口同时出现在训练和验证中。


In [ ]:
def split_melodies_by_piece(melody_sequences, validation_ratio=VALIDATION_RATIO):
    """按旋律条目划分训练/验证，避免窗口级数据泄漏。"""
    shuffled_sequences = list(melody_sequences)
    random.Random(SEED).shuffle(shuffled_sequences)
    if len(shuffled_sequences) < 2:
        return shuffled_sequences, []
    validation_count = max(1, int(round(len(shuffled_sequences) * validation_ratio)))
    validation_sequences = shuffled_sequences[:validation_count]
    training_sequences = shuffled_sequences[validation_count:]
    return training_sequences, validation_sequences


def make_window_examples(melody_sequences, token_to_id, window_length=WINDOW_LENGTH, stride=WINDOW_STRIDE):
    """把 token 序列切成 (context_ids, target_id) 训练样本。"""
    context_windows = []
    target_tokens = []
    for sequence in melody_sequences:
        token_ids = [token_to_id[token] for token in sequence.tokens if token in token_to_id]
        if len(token_ids) <= window_length:
            continue
        for start_index in range(0, len(token_ids) - window_length, stride):
            context_windows.append(token_ids[start_index:start_index + window_length])
            target_tokens.append(token_ids[start_index + window_length])
    return np.asarray(context_windows, dtype=np.int64), np.asarray(target_tokens, dtype=np.int64)


class MelodyWindowDataset(Dataset):
    """PyTorch 数据集：一个样本是一段上下文和下一个 token。"""
    def __init__(self, context_windows, target_tokens):
        self.context_windows = torch.as_tensor(context_windows, dtype=torch.long)
        self.target_tokens = torch.as_tensor(target_tokens, dtype=torch.long)

    def __len__(self):
        return len(self.target_tokens)

    def __getitem__(self, index):
        return self.context_windows[index], self.target_tokens[index]


style_to_data = {}
for style_label, melody_sequences in style_to_melodies.items():
    training_sequences, validation_sequences = split_melodies_by_piece(melody_sequences)
    train_contexts, train_targets = make_window_examples(training_sequences, token_to_id)
    validation_contexts, validation_targets = make_window_examples(validation_sequences, token_to_id)

    style_to_data[style_label] = {
        'training_sequences': training_sequences,
        'validation_sequences': validation_sequences,
        'train_contexts': train_contexts,
        'train_targets': train_targets,
        'validation_contexts': validation_contexts,
        'validation_targets': validation_targets,
    }

window_summary_rows = []
for style_label, data_bundle in style_to_data.items():
    window_summary_rows.append({
        '风格子集': STYLE_DISPLAY_NAMES[style_label],
        '训练曲目数': len(data_bundle['training_sequences']),
        '验证曲目数': len(data_bundle['validation_sequences']),
        '训练窗口数': len(data_bundle['train_targets']),
        '验证窗口数': len(data_bundle['validation_targets']),
    })

display(pd.DataFrame(window_summary_rows))


## 5. 模型：简单 RNN 与 LSTM

此处定义两个结构相近的模型：

- `BasicRNNNextTokenModel`：用于理解“隐藏状态逐步传递”的简单 RNN。
- `LSTMNextTokenModel`：后续风格对比与五声约束采样默认使用该模型。

两者都完成同一个任务：输入一段 token id，输出下一个 token 的 logits。


In [ ]:
class BasicRNNNextTokenModel(nn.Module):
    """简单 RNN：用最后一步隐藏状态预测下一个 token。"""
    def __init__(self, vocabulary_size, embedding_dim, hidden_size, dropout_rate=0.0):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dim)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            nonlinearity='tanh',
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(hidden_size, vocabulary_size)

    def forward(self, token_ids):
        embedded_tokens = self.embedding(token_ids)
        rnn_outputs, _ = self.rnn(embedded_tokens)
        last_hidden_state = rnn_outputs[:, -1, :]
        logits = self.output_layer(self.dropout(last_hidden_state))
        return logits


class LSTMNextTokenModel(nn.Module):
    """LSTM：用门控机制管理长期信息，再预测下一个 token。"""
    def __init__(self, vocabulary_size, embedding_dim, hidden_size, dropout_rate=0.0):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(hidden_size, vocabulary_size)

    def forward(self, token_ids):
        embedded_tokens = self.embedding(token_ids)
        lstm_outputs, _ = self.lstm(embedded_tokens)
        last_hidden_state = lstm_outputs[:, -1, :]
        logits = self.output_layer(self.dropout(last_hidden_state))
        return logits


def create_next_token_model(model_kind, vocabulary_size):
    """根据配置创建简单 RNN 或 LSTM。"""
    if model_kind.lower() == 'rnn':
        return BasicRNNNextTokenModel(
            vocabulary_size=vocabulary_size,
            embedding_dim=EMBEDDING_DIM,
            hidden_size=HIDDEN_SIZE,
            dropout_rate=DROPOUT_RATE,
        )
    if model_kind.lower() == 'lstm':
        return LSTMNextTokenModel(
            vocabulary_size=vocabulary_size,
            embedding_dim=EMBEDDING_DIM,
            hidden_size=HIDDEN_SIZE,
            dropout_rate=DROPOUT_RATE,
        )
    raise ValueError("MODEL_KIND 只能是 'rnn' 或 'lstm'")

print(f'将使用模型: {MODEL_KIND.upper()}')
print('词表大小:', len(token_to_id))


## 6. 训练两个风格模型

这里分别训练两个模型：

1. `ClassicalRomantic-LSTM`：从古典/浪漫旋律子集学习 next-token 预测。
2. `PopLead-LSTM`：从流行主旋律子集学习 next-token 预测。

如果验证窗口数太少，代码仍会训练模型，但验证曲线可能不稳定；因此不能用少量验证窗口概括模型的总体泛化性能。

**图题：两个风格子集上的 LSTM 训练与验证损失。**


In [ ]:
def make_data_loader(context_windows, target_tokens, batch_size, shuffle):
    """创建 DataLoader；如果样本为空则返回 None。"""
    if len(target_tokens) == 0:
        return None
    dataset = MelodyWindowDataset(context_windows, target_tokens)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def run_one_epoch(model, data_loader, loss_function, optimizer=None):
    """运行一个训练或验证 epoch；optimizer 为 None 时只评估。"""
    if data_loader is None:
        return math.nan

    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_examples = 0

    for context_batch, target_batch in data_loader:
        context_batch = context_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        if is_training:
            optimizer.zero_grad()

        logits = model(context_batch)
        loss = loss_function(logits, target_batch)

        if is_training:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        batch_size = target_batch.size(0)
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size

    return total_loss / max(1, total_examples)


def train_style_model(style_label, data_bundle):
    """训练一个风格子集的 next-token 模型。"""
    train_loader = make_data_loader(
        data_bundle['train_contexts'],
        data_bundle['train_targets'],
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    validation_loader = make_data_loader(
        data_bundle['validation_contexts'],
        data_bundle['validation_targets'],
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    if train_loader is None:
        raise RuntimeError(f'{STYLE_DISPLAY_NAMES[style_label]} 没有足够训练窗口。请增加 MAX_FILES_PER_STYLE 或放宽筛选阈值。')

    model = create_next_token_model(MODEL_KIND, vocabulary_size=len(token_to_id)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_function = nn.CrossEntropyLoss()

    history = []
    print(f"\n开始训练: {STYLE_DISPLAY_NAMES[style_label]} ({MODEL_KIND.upper()})")
    for epoch_index in range(1, TRAINING_EPOCHS + 1):
        train_loss = run_one_epoch(model, train_loader, loss_function, optimizer=optimizer)
        validation_loss = run_one_epoch(model, validation_loader, loss_function, optimizer=None)
        history.append({
            'epoch': epoch_index,
            'train_loss': train_loss,
            'validation_loss': validation_loss,
        })
        if math.isnan(validation_loss):
            print(f'  epoch {epoch_index:02d}: train_loss={train_loss:.4f}')
        else:
            print(f'  epoch {epoch_index:02d}: train_loss={train_loss:.4f}, validation_loss={validation_loss:.4f}')

    return model, pd.DataFrame(history)


style_to_model = {}
style_to_history = {}
for style_label, data_bundle in style_to_data.items():
    model, history = train_style_model(style_label, data_bundle)
    style_to_model[style_label] = model
    style_to_history[style_label] = history


In [ ]:
def plot_training_curves(style_to_history):
    """绘制训练/验证 loss 曲线，全部使用灰度和线型区分。"""
    fig, axis = plt.subplots(figsize=(7.5, 4.2))
    style_formats = {
        'classical_romantic': {'color': '0.15', 'linestyle': '-', 'marker': 'o'},
        'pop_lead': {'color': '0.45', 'linestyle': '--', 'marker': 's'},
    }
    for style_label, history in style_to_history.items():
        line_style = style_formats.get(style_label, {'color': '0.25', 'linestyle': '-', 'marker': 'o'})
        axis.plot(
            history['epoch'], history['train_loss'],
            label=f"{STYLE_DISPLAY_NAMES[style_label]}—训练",
            color=line_style['color'], linestyle=line_style['linestyle'], marker=line_style['marker'],
        )
        if not history['validation_loss'].isna().all():
            axis.plot(
                history['epoch'], history['validation_loss'],
                label=f"{STYLE_DISPLAY_NAMES[style_label]}—验证",
                color=line_style['color'], linestyle=':', marker=line_style['marker'], alpha=0.9,
            )
    axis.set_xlabel('训练轮次')
    axis.set_ylabel('交叉熵损失')
    axis.grid(True, color='0.85', linewidth=0.8)
    axis.legend(frameon=False)
    output_path = FIGURES_DIR / 'fig_lstm_training_curves.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('图片已保存:', output_path)

plot_training_curves(style_to_history)


## 7. 采样生成：自由采样与五声音阶约束

生成时，模型输出 logits。我们用温度参数得到概率，再根据需要施加五声音阶掩码。

- **自由采样**：不限制音级（pitch class）。
- **五声约束采样**：只允许 C、D、E、G、A 五个音级；休止仍允许。

五声约束放在采样阶段，因此它改变条件采样分布，但不会更新模型参数。自由与约束版本使用各自独立、可复现且成对相同种子的随机数生成器；若合法概率质量为零，代码会报错而不会退回无约束分布。


In [ ]:
def find_replacement_token(preferred_token, token_counter):
    """若起始 token 不在词表中，寻找一个接近的替代 token。"""
    if preferred_token in token_to_id:
        return preferred_token

    preferred_pitch, preferred_duration = preferred_token
    same_pitch_tokens = [
        token for token in token_counter
        if token[0] == preferred_pitch
    ]
    if same_pitch_tokens:
        return max(same_pitch_tokens, key=lambda token: token_counter[token])

    if preferred_pitch is not REST_PITCH:
        same_pitch_class_tokens = [
            token for token in token_counter
            if token[0] is not REST_PITCH and token[0] % 12 == preferred_pitch % 12
        ]
        if same_pitch_class_tokens:
            return max(same_pitch_class_tokens, key=lambda token: token_counter[token])

    return token_counter.most_common(1)[0][0]


def prepare_seed_tokens(seed_tokens, token_counter):
    """把中性起始动机替换为词表中真实存在的 token。"""
    return [find_replacement_token(token, token_counter) for token in seed_tokens]


def sample_token_id_from_logits(logits, rng, temperature=1.0, allowed_pitch_classes=None):
    """从 logits 中采样一个 token id，可选 pitch-class 约束。"""
    temperature = float(temperature)
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError('temperature 必须是有限正数。')
    if not isinstance(logits, torch.Tensor):
        raise ValueError('logits 必须是 PyTorch Tensor。')
    if logits.ndim != 1 or logits.numel() != len(id_to_token):
        raise ValueError('logits 必须是一维向量，长度等于词表大小。')
    if not bool(torch.isfinite(logits).all()):
        raise ValueError('logits 必须只包含有限数。')
    scaled_logits = logits.detach().cpu().float() / temperature
    if not bool(torch.isfinite(scaled_logits).all()):
        raise ValueError('logits / temperature 必须保持有限。')
    probabilities = torch.softmax(scaled_logits, dim=-1).numpy()

    if allowed_pitch_classes is not None:
        mask = np.array([
            token_allowed_by_pentatonic(id_to_token[token_id], allowed_pitch_classes)
            for token_id in range(len(id_to_token))
        ], dtype=float)
        masked_probabilities = probabilities * mask
        masked_sum = masked_probabilities.sum()
        if not np.isfinite(masked_sum) or masked_sum <= 0:
            raise ValueError('词表中没有满足硬约束且概率非零的 token。')
        probabilities = masked_probabilities / masked_sum

    return int(rng.choice(np.arange(len(probabilities)), p=probabilities))


def generate_token_sequence(model, seed_tokens, generated_token_count, temperature=1.0,
                            allowed_pitch_classes=None, random_seed=SEED):
    """用训练好的模型续写 token 序列。"""
    if not _is_integer_scalar(generated_token_count) or int(generated_token_count) < 0:
        raise ValueError('generated_token_count 必须是非负整数。')
    generated_token_count = int(generated_token_count)
    generated_tokens = list(seed_tokens)
    if not generated_tokens:
        raise ValueError('seed_tokens 不能为空。')
    if any(token not in token_to_id for token in generated_tokens):
        raise ValueError('seed_tokens 中的每个 token 都必须属于词表。')
    model.eval()
    rng = np.random.default_rng(random_seed)

    with torch.no_grad():
        for _ in range(generated_token_count):
            context_tokens = generated_tokens[-WINDOW_LENGTH:]
            if len(context_tokens) < WINDOW_LENGTH:
                padding_count = WINDOW_LENGTH - len(context_tokens)
                context_tokens = [context_tokens[0]] * padding_count + context_tokens
            context_ids = torch.tensor(
                [[token_to_id[token] for token in context_tokens]],
                dtype=torch.long,
                device=DEVICE,
            )
            logits = model(context_ids)[0]
            next_token_id = sample_token_id_from_logits(
                logits,
                rng=rng,
                temperature=temperature,
                allowed_pitch_classes=allowed_pitch_classes,
            )
            generated_tokens.append(id_to_token[next_token_id])

    return generated_tokens


NEUTRAL_SEED_TOKENS = [
    (60, 4),  # C4 quarter
    (62, 4),  # D4 quarter
    (64, 8),  # E4 half
    (67, 4),  # G4 quarter
    (64, 4),  # E4 quarter
    (62, 8),  # D4 half
]
seed_tokens = prepare_seed_tokens(NEUTRAL_SEED_TOKENS, token_counter)
_sampling_test_logits = torch.zeros(len(id_to_token), dtype=torch.float32)
_sampling_test_rng = np.random.default_rng(SEED)
_sampling_test_id = sample_token_id_from_logits(
    _sampling_test_logits, _sampling_test_rng, temperature=1.0
)
assert 0 <= _sampling_test_id < len(id_to_token)
_expect_value_error(
    sample_token_id_from_logits, _sampling_test_logits, np.random.default_rng(SEED),
    temperature=float('nan')
)
_nan_sampling_logits = _sampling_test_logits.clone()
_nan_sampling_logits[0] = float('nan')
_expect_value_error(
    sample_token_id_from_logits, _nan_sampling_logits, np.random.default_rng(SEED)
)
_test_generation_model = next(iter(style_to_model.values()))
assert generate_token_sequence(
    _test_generation_model, seed_tokens, 0, random_seed=SEED
) == seed_tokens
_expect_value_error(generate_token_sequence, _test_generation_model, [], 1)
_expect_value_error(generate_token_sequence, _test_generation_model, seed_tokens, -1)
_expect_value_error(generate_token_sequence, _test_generation_model, seed_tokens, 1.5)
print('实际使用的起始动机:')
print([token_to_text(token) for token in seed_tokens])


In [ ]:
def tokens_to_note_events(tokens, tempo=DEFAULT_TEMPO):
    """将 token 序列转为便于写 MIDI 与绘图的 note event。"""
    seconds_per_beat = 60.0 / tempo
    seconds_per_unit = seconds_per_beat * QUANTIZATION_STEP_BEATS
    current_time = 0.0
    note_events = []
    for token in tokens:
        pitch, duration_units = token
        duration_seconds = max(1, int(duration_units)) * seconds_per_unit
        if pitch is not REST_PITCH:
            note_events.append({
                'pitch': int(pitch),
                'start': current_time,
                'end': current_time + duration_seconds,
                'duration_units': int(duration_units),
            })
        current_time += duration_seconds
    return note_events


def write_tokens_to_midi(tokens, output_path, tempo=DEFAULT_TEMPO, velocity=82):
    """把 token 序列写为钢琴 MIDI。"""
    midi_object = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    piano = pretty_midi.Instrument(program=0, name='Generated Piano Melody')
    for event in tokens_to_note_events(tokens, tempo=tempo):
        piano.notes.append(pretty_midi.Note(
            velocity=velocity,
            pitch=event['pitch'],
            start=event['start'],
            end=event['end'],
        ))
    midi_object.instruments.append(piano)
    midi_object.write(str(output_path))
    return output_path


def show_midi_links(midi_paths):
    """在 Notebook 中显示 MIDI 文件链接。"""
    link_lines = ['生成 MIDI 文件：']
    for midi_path in midi_paths:
        link_lines.append(f'- [{midi_path.name}]({midi_path.as_posix()})')
    display(Markdown('\n'.join(link_lines)))


def preview_midi_audio(midi_path, sample_rate=16000):
    """用 pretty_midi 的简单合成器生成音频预览，方便直接听。"""
    midi_object = pretty_midi.PrettyMIDI(str(midi_path))
    audio = midi_object.synthesize(fs=sample_rate)
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.9
    display(Markdown(f'**音频预览：{midi_path.name}**'))
    display(Audio(audio, rate=sample_rate))

print('MIDI 写入与音频预览函数定义完毕。')


In [ ]:
generation_results = {}
main_midi_paths = []

for style_index, (style_label, model) in enumerate(style_to_model.items()):
    paired_seed = SEED + 1000 + style_index
    free_tokens = generate_token_sequence(
        model=model,
        seed_tokens=seed_tokens,
        generated_token_count=GENERATED_TOKEN_COUNT,
        temperature=DEFAULT_TEMPERATURE,
        allowed_pitch_classes=None,
        random_seed=paired_seed,
    )
    pentatonic_tokens = generate_token_sequence(
        model=model,
        seed_tokens=seed_tokens,
        generated_token_count=GENERATED_TOKEN_COUNT,
        temperature=DEFAULT_TEMPERATURE,
        allowed_pitch_classes=PENTATONIC_PITCH_CLASSES,
        random_seed=paired_seed,
    )

    generation_results[(style_label, 'free')] = free_tokens
    generation_results[(style_label, 'pentatonic')] = pentatonic_tokens

    free_path = MIDI_OUTPUT_DIR / f'{style_label}_free.mid'
    pentatonic_path = MIDI_OUTPUT_DIR / f'{style_label}_pentatonic.mid'
    write_tokens_to_midi(free_tokens, free_path)
    write_tokens_to_midi(pentatonic_tokens, pentatonic_path)
    main_midi_paths.extend([free_path, pentatonic_path])

show_midi_links(main_midi_paths)

if RENDER_AUDIO_PREVIEW:
    for midi_path in main_midi_paths:
        preview_midi_audio(midi_path)


## 8. 灰度钢琴卷帘图：四个主输出

1. 古典/浪漫模型：自由采样
2. 古典/浪漫模型：五声约束
3. 流行主旋律模型：自由采样
4. 流行主旋律模型：五声约束


In [ ]:
def plot_piano_roll_for_tokens(axis, tokens, title):
    """用灰度线段绘制 token 序列的钢琴卷帘。"""
    note_events = tokens_to_note_events(tokens)
    for event in note_events:
        axis.hlines(
            y=event['pitch'],
            xmin=event['start'],
            xmax=event['end'],
            color='0.15',
            linewidth=3.0,
        )
    axis.set_title(title)
    axis.set_xlabel('时间（秒）')
    axis.set_ylabel('MIDI 音高')
    axis.grid(True, color='0.88', linewidth=0.7)
    if note_events:
        pitches = [event['pitch'] for event in note_events]
        axis.set_ylim(min(pitches) - 3, max(pitches) + 3)


def plot_generation_piano_rolls(generation_results):
    subplot_specs = [
        ('classical_romantic', 'free', '古典/浪漫：自由采样'),
        ('classical_romantic', 'pentatonic', '古典/浪漫：五声约束'),
        ('pop_lead', 'free', '流行主旋律：自由采样'),
        ('pop_lead', 'pentatonic', '流行主旋律：五声约束'),
    ]
    fig, axes = plt.subplots(4, 1, figsize=(10, 8.5), sharex=True, constrained_layout=True)
    for axis, (style_label, sampling_label, title) in zip(axes, subplot_specs):
        plot_piano_roll_for_tokens(axis, generation_results[(style_label, sampling_label)], title)
    output_path = FIGURES_DIR / 'fig_lstm_generation_pianorolls.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('图片已保存:', output_path)

plot_generation_piano_rolls(generation_results)


## 9. 温度参数实验

温度改变采样分布，而不是改变模型参数：

- `T < 1`：在 logits 不全相等时使概率更尖锐；序列是否更保守或更重复只是常见经验趋势，不是数学保证。
- `T = 1`：使用模型原本的分布。
- `T > 1`：在 logits 不全相等时使概率更平坦；具体音乐行为仍取决于模型、上下文和随机样本。

下面固定使用流行主旋律模型，比较不同温度的音高轮廓，并输出对应 MIDI。

**图题：不同温度参数下 96 个新生成 token 的音高轮廓（休止处留为空白）。**


In [ ]:
def pitch_series_from_tokens(tokens):
    """按生成 token 顺序提取音高；休止 token 用 NaN 留出空白。"""
    return [np.nan if pitch is REST_PITCH else pitch for pitch, _ in tokens]


temperature_results = {}
temperature_midi_paths = []
reference_style_label = 'pop_lead'
reference_model = style_to_model[reference_style_label]

for temperature in TEMPERATURE_VALUES:
    generated_tokens = generate_token_sequence(
        model=reference_model,
        seed_tokens=seed_tokens,
        generated_token_count=GENERATED_TOKEN_COUNT,
        temperature=temperature,
        allowed_pitch_classes=None,
        random_seed=SEED + 2000,
    )
    temperature_results[temperature] = generated_tokens
    safe_temperature_text = str(temperature).replace('.', '_')
    output_path = MIDI_OUTPUT_DIR / f'temperature_{safe_temperature_text}.mid'
    write_tokens_to_midi(generated_tokens, output_path)
    temperature_midi_paths.append(output_path)

show_midi_links(temperature_midi_paths)

fig, axis = plt.subplots(figsize=(9, 4.2))
line_styles = ['-', '--', ':']
line_colors = ['0.10', '0.35', '0.60']
for line_index, temperature in enumerate(TEMPERATURE_VALUES):
    continuation = temperature_results[temperature][len(seed_tokens):]
    pitch_series = pitch_series_from_tokens(continuation)
    generation_steps = np.arange(1, len(pitch_series) + 1)
    axis.plot(
        generation_steps,
        pitch_series,
        label=f'T={temperature}',
        color=line_colors[line_index % len(line_colors)],
        linestyle=line_styles[line_index % len(line_styles)],
        linewidth=1.8,
    )
axis.set_xlim(1, GENERATED_TOKEN_COUNT)
axis.set_xlabel('新生成 token 的序号')
axis.set_ylabel('MIDI 音高')
axis.grid(True, color='0.85', linewidth=0.8)
axis.legend(frameon=False)
output_path = FIGURES_DIR / 'fig_temperature_contours.png'
fig.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()
print('图片已保存:', output_path)


## 10. 生成结果统计：音级、音程、时值与五声合规率

统计图用于把听觉观察与可计算特征对应起来。例如：

- 五声约束是否真的提高了五声合规率？
- 两个风格模型的重复率、音域、时值分布是否不同？
- 统计上更接近五声音阶，并不等于自动获得中国音乐的腔韵、句法与演奏法。


In [ ]:
def note_pitches_from_tokens(tokens):
    return [pitch for pitch, _ in tokens if pitch is not REST_PITCH]


def durations_from_tokens(tokens):
    return [duration_units for _, duration_units in tokens]


def pitch_class_counts(tokens):
    counter = Counter(pitch % 12 for pitch in note_pitches_from_tokens(tokens))
    return np.array([counter.get(pitch_class, 0) for pitch_class in range(12)], dtype=float)


def interval_counts(tokens):
    pitches = note_pitches_from_tokens(tokens)
    intervals = [pitches[index + 1] - pitches[index] for index in range(len(pitches) - 1)]
    if not intervals:
        return [], np.array([], dtype=float)
    bins = list(range(min(intervals), max(intervals) + 1))
    counter = Counter(intervals)
    return bins, np.array([counter.get(interval, 0) for interval in bins], dtype=float)


def duration_counts(tokens):
    durations = [int(duration) for duration in durations_from_tokens(tokens)]
    if not durations:
        return [], np.array([], dtype=float)
    counter = Counter(durations)
    bins = list(range(min(durations), max(durations) + 1))
    return bins, np.array([counter.get(duration, 0) for duration in bins], dtype=float)


def normalized(values):
    values = np.asarray(values, dtype=float)
    total = values.sum()
    if total <= 0:
        return values
    return values / total


def ngram_repetition_rate(tokens, ngram_size=2):
    if len(tokens) < ngram_size:
        return 0.0
    ngrams = [tuple(tokens[index:index + ngram_size]) for index in range(len(tokens) - ngram_size + 1)]
    unique_count = len(set(ngrams))
    return 1.0 - unique_count / len(ngrams)


def pentatonic_compliance_rate(tokens, allowed_pitch_classes=PENTATONIC_PITCH_CLASSES):
    pitches = note_pitches_from_tokens(tokens)
    if not pitches:
        return 0.0
    allowed_count = sum(1 for pitch in pitches if pitch % 12 in allowed_pitch_classes)
    return allowed_count / len(pitches)


def summarize_generated_tokens(label, tokens):
    pitches = note_pitches_from_tokens(tokens)
    rest_count = sum(1 for pitch, _ in tokens if pitch is REST_PITCH)
    return {
        '生成版本': label,
        '总 token 数': len(tokens),
        '种子 token 数': len(seed_tokens),
        '新生成 token 数': max(0, len(tokens) - len(seed_tokens)),
        '音符数': len(pitches),
        '休止比例': round(rest_count / max(1, len(tokens)), 3),
        '音域跨度': int(max(pitches) - min(pitches)) if pitches else 0,
        '平均音高': round(float(np.mean(pitches)), 1) if pitches else 0,
        '2-gram重复率': round(ngram_repetition_rate(tokens, 2), 3),
        '3-gram重复率': round(ngram_repetition_rate(tokens, 3), 3),
        '五声合规率': round(pentatonic_compliance_rate(tokens), 3),
    }

metric_rows = []
for style_label in ['classical_romantic', 'pop_lead']:
    for sampling_label, sampling_name in [('free', '自由'), ('pentatonic', '五声')]:
        readable_label = f"{STYLE_DISPLAY_NAMES[style_label]}-{sampling_name}"
        metric_rows.append(summarize_generated_tokens(readable_label, generation_results[(style_label, sampling_label)]))

metric_table = pd.DataFrame(metric_rows)
display(metric_table)


In [ ]:
def plot_generation_statistics(generation_results):
    """绘制音级、音程、时值三组统计图。"""
    series_specs = [
        ('classical_romantic', 'free', '古典自由', '0.15', ''),
        ('classical_romantic', 'pentatonic', '古典五声', '0.35', '//'),
        ('pop_lead', 'free', '流行自由', '0.60', ''),
        ('pop_lead', 'pentatonic', '流行五声', '0.80', 'xx'),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

    # 音级分布
    x_positions = np.arange(12)
    bar_width = 0.18
    for series_index, (style_label, sampling_label, label, color, hatch) in enumerate(series_specs):
        values = normalized(pitch_class_counts(generation_results[(style_label, sampling_label)]))
        axes[0].bar(
            x_positions + (series_index - 1.5) * bar_width,
            values,
            width=bar_width,
            label=label,
            color=color,
            edgecolor='black',
            hatch=hatch,
        )
    axes[0].set_title('音级分布')
    axes[0].set_xticks(x_positions)
    axes[0].set_xticklabels(PC_NAMES)
    axes[0].set_ylabel('占比')

    # interval distribution
    for style_label, sampling_label, label, color, hatch in series_specs:
        bins, values = interval_counts(generation_results[(style_label, sampling_label)])
        axes[1].plot(bins, normalized(values), label=label, color=color, linewidth=1.8)
    axes[1].set_title('音程分布')
    axes[1].set_xlabel('音程（半音）')
    axes[1].set_ylabel('占比')

    # duration distribution
    for series_index, (style_label, sampling_label, label, color, hatch) in enumerate(series_specs):
        bins, values = duration_counts(generation_results[(style_label, sampling_label)])
        axes[2].bar(
            np.array(bins) + (series_index - 1.5) * bar_width,
            normalized(values),
            width=bar_width,
            label=label,
            color=color,
            edgecolor='black',
            hatch=hatch,
        )
    axes[2].set_title('时值分布')
    axes[2].set_xlabel('时值单位')
    axes[2].set_ylabel('占比')

    for axis in axes:
        axis.grid(True, axis='y', color='0.86', linewidth=0.8)
        axis.set_axisbelow(True)
    axes[0].legend(frameon=False, fontsize=9)

    output_path = FIGURES_DIR / 'fig_lstm_generation_stats.png'
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('图片已保存:', output_path)

plot_generation_statistics(generation_results)


## 11. 可改参数练习

建议复制此 Notebook 后做以下实验：

1. 把 `DEFAULT_TEMPERATURE` 改成 `0.5`、`1.0` 或 `2.0`，比较概率分布、重复率与生成轨迹。
2. 把 `PENTATONIC_PITCH_CLASSES` 改成 `{2, 4, 6, 9, 11}`，观察 D 宫五声约束下的变化。
3. 把 `MODEL_KIND` 改成 `'rnn'`，比较简单 RNN 与 LSTM 的训练曲线和生成 token 统计。
4. 改写 `NEUTRAL_SEED_TOKENS`，例如用《茉莉花》开头作为 seed，但注意这只是起始动机，不代表模型学习了《茉莉花》。
5. 增大 `MAX_FILES_PER_STYLE` 和 `TRAINING_EPOCHS`，记录训练/验证损失与重复率如何变化；单次变化不能单独归因于其中一个参数。

思考：五声音阶约束后的结果是否“更像中国音乐”？如果只是更少出现半音，它距离中国音乐作品中的腔韵、句法、装饰音和调式重心还差什么？
